# 05 Training
Trains every network listed under `variants` in `config.yaml`. Uses all GPUs (and all `nodes`) it finds; a run that was interrupted resumes from its last checkpoint.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
from retinopath.jobs import run_jobs, train_commands, n_gpus
jobs = train_commands(cfg)
print(len(jobs), "training runs:", [name for name, _ in jobs]); print("GPUs here:", n_gpus(), "| nodes:", list(cfg.nodes) or "none", "| mode:", cfg.multi_gpu)
with Timer(cfg, "05_training"):
    done = run_jobs(cfg, jobs)
assert all(return_code == 0 for _, return_code in done), "a training run failed: see outputs/logs"

In [ ]:
from retinopath.core import load_json
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, _ in jobs:
    log_table = pd.DataFrame(load_json(cfg.P.out / "models" / name / "log.json"))
    if len(log_table):
        axes[0].plot(log_table.step, log_table.val_voxel, label=name); axes[1].plot(log_table.step, log_table.val_path_mm, label=name)
axes[0].set_title("validation voxel loss"); axes[1].set_title("validation path error (mm)"); axes[1].legend(fontsize=7)
[axis.set_xlabel("step") for axis in axes]; fig.savefig(FIG / "training_curves.png", dpi=150, bbox_inches="tight")